# Capstone — Which pages should a content team refresh first?

This notebook mirrors the deployed research paper section by section. It is the single, rerunnable source of every number and figure the paper shows — run it top to bottom and it regenerates `docs/figures/*.svg` and the ranked queue in `work/outputs/`.

Lane: **Refresh / Content Opportunity Scoring** · Data: FlyRank ML Internship starter release (30,000 pseudonymized pages, 32 clients).


## 1. Question

**Decision:** out of thousands of published pages, which should a content editor fix first, given limited review capacity? **Output:** a ranked queue with reason codes and actions. **Cost of a wrong call:** a false positive burns editor hours; a false negative lets a decaying page keep losing visibility. **Why ML:** the pattern (staleness + demand + CTR + position + depth) is real but too tangled for one if-statement — measured in Week 4–6.


In [1]:
# The decision, as measurable framing
import pandas as pd

framing = pd.DataFrame([
    {"axis": "Task type", "answer": "Ranking / scoring (a classifier's probability, ranked)"},
    {"axis": "Primary metric", "answer": "Precision@50 (matches a reviewer's real capacity)"},
    {"axis": "Base rate to beat", "answer": "54.2% declining overall; 39.1% on held-out clients"},
    {"axis": "Honest split", "answer": "client-holdout (grouped by client_id)"},
    {"axis": "Claim language", "answer": "observed / measured / directional / decision-support"},
])
framing


,axis,answer
0,Task type,"Ranking / scoring (a classifier's probability,..."
1,Primary metric,Precision@50 (matches a reviewer's real capacity)
2,Base rate to beat,54.2% declining overall; 39.1% on held-out cli...
3,Honest split,client-holdout (grouped by client_id)
4,Claim language,observed / measured / directional / decision-s...


## 2. Data

**Release:** the bundled starter slice `data/raw/content_refresh_anonymized.csv` — 30,000 rows × 44 columns, one row per pseudonymized content item, 32 pseudonymized clients, trailing-90-day metrics. (The ~79M-row warehouse release is the forward-window next step, not what is modeled here.)

**Excluded, and why:** `trend_direction` / `trend_pct` (label source), the 30-day comparison columns (trend inputs), `days_with_impressions` / `days_with_sessions` (same-window presence counters), pseudonymous IDs (grouping only), and provider/model metadata.


In [2]:
# ---- Load the data and state the grain + label ----
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(os.getcwd())
while not (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
df["declining"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

print(f"rows: {len(df):,}   columns: {df.shape[1]}   clients: {df['client_id'].nunique()}")
print(f"label (declining = trend_direction == 'down'): {df['declining'].mean():.3f} "
      f"({int(df['declining'].sum()):,} of {len(df):,})")
print("\nexcluded columns never used as features:")
print("  trend_direction, trend_pct, impressions/clicks/sessions_last_30d + _prev_30d,")
print("  days_with_impressions, days_with_sessions, content_id, client_id, provider_used, model_used")


rows: 30,000   columns: 45   clients: 32
label (declining = trend_direction == 'down'): 0.542 (16,262 of 30,000)

excluded columns never used as features:
  trend_direction, trend_pct, impressions/clicks/sessions_last_30d + _prev_30d,
  days_with_impressions, days_with_sessions, content_id, client_id, provider_used, model_used


## 3. Methodology

- **Label:** `is_declining_label = (trend_direction == "down")` — a same-window proxy, not a future outcome.
- **Baseline (Week-4 rule):** `score = stale(≥90d) × demand(percentile of log impressions) × ctr_weak(1 − min(ctr,2)/2)`.
- **Models:** logistic regression, depth-limited decision tree, random forest, histogram gradient boosting — ranked by predicted probability.
- **Validation:** client-holdout (seed 42), ~20% of clients held out.
- **Leakage checks:** confirm `trend_pct` and `days_with_impressions` are excluded because they inflate the score.


In [3]:
# ---- Features, baseline, models, split ----
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

NUMERIC = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

num = df[NUMERIC].apply(pd.to_numeric, errors="coerce")
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "users_90d",
            "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "search_volume"]:
    num[f"log_{col}"] = np.log1p(num[col])
num["has_position"] = (df["avg_position"] > 0).astype(int)
num["has_word_count"] = (df["word_count"] > 0).astype(int)
num = num.replace([np.inf, -np.inf], np.nan).fillna(0)
cat = df[CATEGORICAL].fillna("unknown").astype(str)
cat_enc = pd.get_dummies(cat, prefix=CATEGORICAL, dtype=float)
X = pd.concat([num.reset_index(drop=True), cat_enc.reset_index(drop=True)], axis=1)
y = df["declining"].to_numpy()

# baseline rule
stale = (df["days_since_last_update"] >= 90).astype(int).to_numpy()
demand = np.log1p(df["impressions_90d"]).rank(method="average", pct=True).fillna(0).to_numpy()
ctr_weak = (1 - df["ctr"].clip(lower=0, upper=2) / 2.0).to_numpy()
baseline_score = stale * demand * ctr_weak

def make_models():
    return {
        "logistic_regression": Pipeline([("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))]),
        "decision_tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=50,
            class_weight="balanced", random_state=RANDOM_STATE),
        "random_forest": RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
            class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
        "hist_gradient_boosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1,
            max_leaf_nodes=31, min_samples_leaf=50, l2_regularization=1.0, random_state=RANDOM_STATE),
    }

def p_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(labels)[order[:k]].mean())

# client-holdout split (honest)
rng = np.random.default_rng(RANDOM_STATE)
clients = df["client_id"].drop_duplicates().to_numpy()
shuffled = rng.permutation(clients)
test_clients = set(shuffled[: max(1, int(round(len(shuffled) * 0.2)))])
test_mask = df["client_id"].isin(test_clients).to_numpy()
tr_hold, te_hold = np.where(~test_mask)[0], np.where(test_mask)[0]
print(f"client-holdout: {len(test_clients)} of {len(clients)} clients held out; "
      f"test {len(te_hold):,} rows, positive rate {y[te_hold].mean():.3f}")

# ---- Leakage checks ----
X_plus_trend = X.copy(); X_plus_trend["trend_pct"] = df["trend_pct"].fillna(0).to_numpy()
lr_leak = Pipeline([("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
lr_leak.fit(X_plus_trend.iloc[tr_hold], y[tr_hold])
leak_p = lr_leak.predict_proba(X_plus_trend.iloc[te_hold])[:, 1]
print(f"leak check — adding trend_pct (label source): AUC = {roc_auc_score(y[te_hold], leak_p):.3f} (excluded)")

single = pd.DataFrame({"days_with_impressions": df["days_with_impressions"].to_numpy()})
hgb1 = HistGradientBoostingClassifier(max_iter=200, random_state=RANDOM_STATE)
hgb1.fit(single.iloc[tr_hold], y[tr_hold])
single_p = hgb1.predict_proba(single.iloc[te_hold])[:, 1]
print(f"leak check — days_with_impressions alone:        AUC = {roc_auc_score(y[te_hold], single_p):.3f} (excluded)")


client-holdout: 6 of 32 clients held out; test 2,325 rows, positive rate 0.391
leak check — adding trend_pct (label source): AUC = 0.999 (excluded)


leak check — days_with_impressions alone:        AUC = 0.758 (excluded)


## 4. Results (vs baseline)

Baseline and all four models scored **only on the held-out clients**, same Precision@K metric. The chart and table below are the paper's headline result.


In [4]:
# ---- Train + compare (client-holdout) ----
import html as _html
from sklearn.metrics import average_precision_score

def make_lr():
    return Pipeline([("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])

rows = []
for name, template in make_models().items():
    import copy
    m = copy.deepcopy(template)
    m.fit(X.iloc[tr_hold], y[tr_hold])
    p = m.predict_proba(X.iloc[te_hold])[:, 1]
    rows.append({
        "model": name,
        "P@20": round(p_at_k(p, y[te_hold], 20), 3),
        "P@50": round(p_at_k(p, y[te_hold], 50), 3),
        "P@100": round(p_at_k(p, y[te_hold], 100), 3),
        "ROC-AUC": round(float(roc_auc_score(y[te_hold], p)), 3),
        "Avg precision": round(float(average_precision_score(y[te_hold], p)), 3),
    })

baseline_row = {
    "model": "baseline (W4 rule)",
    "P@20": round(p_at_k(baseline_score[te_hold], y[te_hold], 20), 3),
    "P@50": round(p_at_k(baseline_score[te_hold], y[te_hold], 50), 3),
    "P@100": round(p_at_k(baseline_score[te_hold], y[te_hold], 100), 3),
    "ROC-AUC": round(float(roc_auc_score(y[te_hold], baseline_score[te_hold])), 3),
    "Avg precision": round(float(average_precision_score(y[te_hold], baseline_score[te_hold])), 3),
}
comparison = pd.DataFrame([baseline_row] + rows)
print(f"base rate on held-out clients: {y[te_hold].mean():.3f}")
print(comparison.to_string(index=False))

# ---- Random-split before/after for the validation chart ----
idx = np.arange(len(df))
tr_rand, te_rand = train_test_split(idx, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
rand_rows = {}
for name, template in make_models().items():
    import copy
    m = copy.deepcopy(template)
    m.fit(X.iloc[tr_rand], y[tr_rand])
    rand_rows[name] = p_at_k(m.predict_proba(X.iloc[te_rand])[:, 1], y[te_rand], 50)
hold_rows = {r["model"]: r["P@50"] for r in rows}
random_vs_hold = pd.DataFrame([
    {"model": n, "P@50 random split": round(rand_rows[n], 3), "P@50 client holdout": hold_rows[n]} for n in rand_rows
])
print("\nbefore/after (split honesty):")
print(random_vs_hold.to_string(index=False))

# ---- Figures ----
FIG = ROOT / "docs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

def esc(s): return _html.escape(str(s))

def svg_model_vs_baseline(path):
    bars = [
        ("Baseline (W4 rule)", baseline_row["P@50"], "#9aa5ad"),
        ("Decision tree", rows[0]["P@50"], "#426B69"),
        ("Random forest", rows[1]["P@50"], "#426B69"),
        ("Logistic regression", rows[2]["P@50"], "#6F4E7C"),
        ("Gradient boosting", rows[3]["P@50"], "#426B69"),
    ]
    base = float(y[te_hold].mean())
    left, plot_w = 240, 440
    top = 70; row_h = 40
    h = top + len(bars)*row_h + 30
    w = left + plot_w + 90
    def x(v): return left + (v/1.0)*plot_w
    lines = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" viewBox="0 0 {w} {h}">',
             '<rect width="100%" height="100%" fill="#ffffff"/>',
             f'<text x="{left}" y="34" font-family="Arial" font-size="20" font-weight="bold" fill="#16232a">Precision@50 on held-out clients</text>',
             f'<text x="{left}" y="56" font-family="Arial" font-size="13" fill="#5a6670">higher is better — measured on {len(test_clients)} clients the models never trained on</text>']
    bx = x(base)
    lines.append(f'<line x1="{bx:.1f}" y1="{top-8}" x2="{bx:.1f}" y2="{h-14}" stroke="#b07aa1" stroke-width="1.5" stroke-dasharray="5 4"/>')
    lines.append(f'<text x="{bx+6:.1f}" y="{top-2}" font-family="Arial" font-size="12" fill="#b07aa1">test base rate {base:.3f}</text>')
    for i,(lab,v,color) in enumerate(bars):
        yy = top + i*row_h
        lines.append(f'<text x="{left-10}" y="{yy+18}" text-anchor="end" font-family="Arial" font-size="14" fill="#27343b">{esc(lab)}</text>')
        lines.append(f'<rect x="{left}" y="{yy}" width="{x(v)-left:.1f}" height="26" fill="{color}" rx="4"/>')
        lines.append(f'<text x="{x(v)+8:.1f}" y="{yy+18}" font-family="Arial" font-size="13" font-weight="bold" fill="#16232a">{v:.2f}</text>')
    lines.append('</svg>')
    path.write_text("\n".join(lines))

def svg_split(path):
    mods = [(r["model"].replace("_", " ").title(), rand_rows[r["model"]], r["P@50"]) for r in rows]
    g_left, g_plot_w = 250, 480
    g_top = 78; g_row_h = 48
    g_h = g_top + len(mods)*g_row_h + 46
    g_w = g_left + g_plot_w + 90
    group_w = g_plot_w / len(mods); bar_w = 22
    def gx(v): return g_left + (v/1.0)*g_plot_w
    g = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{g_w}" height="{g_h}" viewBox="0 0 {g_w} {g_h}">',
         '<rect width="100%" height="100%" fill="#ffffff"/>',
         f'<text x="{g_left}" y="32" font-family="Arial" font-size="19" font-weight="bold" fill="#16232a">Why the split matters: Precision@50, random vs client-holdout</text>',
         f'<text x="{g_left}" y="54" font-family="Arial" font-size="13" fill="#5a6670">the gap is skill the random split was faking by leaking clients across train/test</text>']
    g.append(f'<rect x="{g_left}" y="{g_top-24}" width="14" height="14" fill="#c9d4db" rx="2"/>')
    g.append(f'<text x="{g_left+20}" y="{g_top-12}" font-family="Arial" font-size="12" fill="#27343b">random row split (naive)</text>')
    g.append(f'<rect x="{g_left+170}" y="{g_top-24}" width="14" height="14" fill="#6F4E7C" rx="2"/>')
    g.append(f'<text x="{g_left+190}" y="{g_top-12}" font-family="Arial" font-size="12" fill="#27343b">client holdout (honest)</text>')
    for i,(lab,vr,vh) in enumerate(mods):
        gy = g_top + i*g_row_h
        cx = g_left + i*group_w + group_w/2
        g.append(f'<text x="{g_left-10}" y="{gy+16}" text-anchor="end" font-family="Arial" font-size="13" fill="#27343b">{esc(lab)}</text>')
        g.append(f'<rect x="{cx-30:.1f}" y="{gy}" width="{bar_w}" height="{gx(vr)-g_left:.1f}" fill="#c9d4db" rx="2"/>')
        g.append(f'<text x="{cx-19:.1f}" y="{gy+gx(vr)-g_left+16:.1f}" font-family="Arial" font-size="11" fill="#27343b">{vr:.2f}</text>')
        g.append(f'<rect x="{cx+6:.1f}" y="{gy}" width="{bar_w}" height="{gx(vh)-g_left:.1f}" fill="#6F4E7C" rx="2"/>')
        g.append(f'<text x="{cx+17:.1f}" y="{gy+gx(vh)-g_left+16:.1f}" font-family="Arial" font-size="11" fill="#27343b">{vh:.2f}</text>')
    g.append('</svg>')
    path.write_text("\n".join(g))

svg_model_vs_baseline(FIG / "model_vs_baseline.svg")
svg_split(FIG / "split_before_after.svg")
print(f"\nwrote {FIG / 'model_vs_baseline.svg'} and {FIG / 'split_before_after.svg'}")


base rate on held-out clients: 0.391
                 model  P@20  P@50  P@100  ROC-AUC  Avg precision
    baseline (W4 rule)   0.5  0.26   0.32    0.493          0.392
   logistic_regression   0.7  0.72   0.71    0.711          0.580
         decision_tree   0.6  0.56   0.60    0.735          0.572
         random_forest   0.8  0.66   0.65    0.735          0.584
hist_gradient_boosting   0.8  0.72   0.69    0.721          0.578



before/after (split honesty):
                 model  P@50 random split  P@50 client holdout
   logistic_regression               0.88                 0.72
         decision_tree               0.90                 0.56
         random_forest               0.90                 0.66
hist_gradient_boosting               0.94                 0.72

wrote /Users/egealgel/Documents/FlyRankAI/flyrank-ml-internship-starter/docs/figures/model_vs_baseline.svg and /Users/egealgel/Documents/FlyRankAI/flyrank-ml-internship-starter/docs/figures/split_before_after.svg


## 5. Limitations

The label is a same-window proxy — "decline risk" is the snapshot's decline state, not a forecast of next month. Precision@50 = 0.72 still means ~1 in 4 top-50 picks is wrong. The queue is cross-client and unscaled (a single high-volume client can dominate the top). No causal claim ("a refresh will lift traffic") is supported without a before/after design. Claims stay in observed / measured / directional / decision-support language.


In [5]:
# ---- Where the model is wrong (error by tier) ----
lr_full = make_lr()
lr_full.fit(X, y)
full_proba = lr_full.predict_proba(X)[:, 1]

te = df[test_mask].copy()
te["proba"] = lr_full.predict_proba(X[test_mask])[:, 1]
te["pred"] = (te["proba"] >= 0.5).astype(int)
te["wrong"] = (te["pred"] != te["declining"]).astype(int)
err = (te.groupby("position_tier", observed=True)
       .agg(n=("content_id", "size"), error_rate=("wrong", "mean"), positive_rate=("declining", "mean"))
       .round(3))
print("error rate by position tier (held-out clients):")
print(err.to_string())


error rate by position tier (held-out clients):
                  n  error_rate  positive_rate
position_tier                                 
deep             60       0.500          0.600
page_1         1061       0.350          0.435
page_3_5        281       0.377          0.488
striking        405       0.422          0.531
top_3           518       0.097          0.114


## 6. Ranked recommendations

The validated model ranks all 30,000 pages; a transparent reason ladder attaches one action per row. The top of the queue is where a reviewer starts; `defer` catches pages with nothing left to recover.


In [6]:
# ---- The action playbook: reason ladder + queue ----
out = df[["content_id", "client_id", "impressions_90d", "clicks_90d", "sessions_90d",
          "ctr", "avg_position", "days_since_last_update", "content_age_days", "word_count",
          "cpc", "content_type"]].copy()
out["action_score"] = np.round(full_proba, 4)
out["captured_value"] = (out["clicks_90d"] * out["cpc"]).round(2)

def reason_action(r):
    pos = r["avg_position"]; imp = r["impressions_90d"]; ctr = r["ctr"]
    stale = r["days_since_last_update"]; wc = r["word_count"]; clicks = r["clicks_90d"]
    if 0 < pos <= 3: return "top3_asset", "protect"
    if stale >= 90 and imp >= 500: return "stale_visible", "refresh"
    if imp >= 500 and ctr < 0.5: return "visible_low_ctr", "fix_ctr"
    if 11 <= pos <= 20 and imp >= 300: return "striking_distance", "optimize_snippet"
    if wc > 0 and wc < 1200 and imp >= 250: return "thin_visible", "expand"
    if clicks == 0 and (pos > 50 or pos == 0): return "deep_no_clicks", "defer"
    return "healthy_or_ambiguous", "monitor"

codes = out.apply(reason_action, axis=1, result_type="expand")
out["reason_code"] = codes[0]
out["action_label"] = codes[1]
out["rank"] = out["action_score"].rank(method="first", ascending=False).astype(int)
queue = out.sort_values("rank").reset_index(drop=True)

print("action mix:")
print(queue["action_label"].value_counts().to_string())
print("\nvalue at stake by action (median impressions):")
print(queue.groupby("action_label")["impressions_90d"].agg(n="size", median="median").round(0).to_string())
print("\ntop 10 of the queue:")
print(queue[["rank","content_id","action_score","action_label","reason_code","impressions_90d","avg_position","ctr"]].head(10).to_string(index=False))


action mix:
action_label
monitor             11228
fix_ctr              8402
refresh              6338
defer                2031
protect              1141
optimize_snippet      819
expand                 41

value at stake by action (median impressions):
                      n  median
action_label                   
defer              2031     3.0
expand               41   345.0
fix_ctr            8402  2509.0
monitor           11228    92.0
optimize_snippet    819   451.0
protect            1141    74.0
refresh            6338  3414.0

top 10 of the queue:
 rank           content_id  action_score action_label          reason_code  impressions_90d  avg_position  ctr
    1 content_f986bd514b6e        0.9456      fix_ctr      visible_low_ctr            22456           6.6 0.00
    2 content_c89e3b5466ba        0.9391      fix_ctr      visible_low_ctr             2321          11.8 0.00
    3 content_c8ad1f4d0e56        0.9376      monitor healthy_or_ambiguous            62927           

## 7. Artifacts the paper embeds

The cell below writes the ranked queue (regenerated, gitignored), the metrics receipt (committed), and the action-mix figure — the exact files the deployed page and the playbook reuse.


In [7]:
# ---- Exports ----
OUT_DIR = ROOT / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

export_cols = ["rank", "content_id", "client_id", "action_score", "action_label", "reason_code",
               "impressions_90d", "clicks_90d", "avg_position", "ctr",
               "days_since_last_update", "content_age_days", "word_count", "content_type", "captured_value"]
queue_path = OUT_DIR / "action_playbook_queue.csv"
queue[export_cols].to_csv(queue_path, index=False)

metrics = {
    "rows": int(len(queue)),
    "validated_precision_at_50_client_holdout": round(float(comparison[comparison.model == "logistic_regression"]["P@50"].iloc[0]), 3),
    "comparison_table": comparison.to_dict(orient="records"),
    "action_mix": queue["action_label"].value_counts().to_dict(),
    "label_base_rate_full": round(float(df["declining"].mean()), 4),
    "label_base_rate_holdout": round(float(y[te_hold].mean()), 4),
}
metrics_path = OUT_DIR / "capstone_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))

# action-mix figure
FIG = ROOT / "docs" / "figures"
mix = queue["action_label"].value_counts()
labels = mix.index.tolist(); values = mix.values.tolist()
mx = max(values) if values else 1; mx = max(mx, 1)
w, ml = 760, 170
h = 54 + 34 * len(values)
lines = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}">',
         '<rect width="100%" height="100%" fill="#ffffff"/>',
         f'<text x="{w/2}" y="26" text-anchor="middle" font-family="Arial" font-size="17" fill="#16232a">Action mix — content playbook</text>']
for i,(lab,v) in enumerate(zip(labels, values)):
    y = 46 + i*34
    bw = (v/mx)*(w-ml-60)
    lines.append(f'<text x="{ml-8}" y="{y+13}" text-anchor="end" font-family="Arial" font-size="13" fill="#27343b">{esc(lab)}</text>')
    lines.append(f'<rect x="{ml}" y="{y}" width="{bw:.1f}" height="22" fill="#426B69" rx="3"/>')
    lines.append(f'<text x="{ml+bw+6:.1f}" y="{y+15}" font-family="Arial" font-size="13" fill="#27343b">{int(v):,}</text>')
lines.append('</svg>')
(FIG / "action_mix_playbook.svg").write_text("\n".join(lines))

print(f"wrote {queue_path}")
print(f"wrote {metrics_path}")
print(f"wrote {FIG / 'action_mix_playbook.svg'}")


wrote /Users/egealgel/Documents/FlyRankAI/flyrank-ml-internship-starter/work/outputs/action_playbook_queue.csv
wrote /Users/egealgel/Documents/FlyRankAI/flyrank-ml-internship-starter/work/outputs/capstone_metrics.json
wrote /Users/egealgel/Documents/FlyRankAI/flyrank-ml-internship-starter/docs/figures/action_mix_playbook.svg


## 8. 5-minute demo outline (optional showcase)

**0:00–0:30 — Question.** Out of thousands of pages, which one should a content editor fix first? FlyRank publishes at scale; pages rank, then quietly decay, and teams notice too late.

**0:30–1:30 — Method.** A transparent hand-written rule first (`stale × demand × ctr_weak`), then a logistic regression that must beat it on the *same data, same split, same metric* — validated on a client-holdout (6 clients the model never trained on).

**1:30–2:00 — One chart.** `docs/figures/model_vs_baseline.svg` — the rule ranks below the base rate on unseen clients; the model clears it.

**2:00–3:00 — One honest result.** On held-out clients the baseline sits at Precision@50 = 0.26 (below the 39.1% base rate); logistic regression reaches 0.72. Still ~1 in 4 top-50 picks is wrong.

**3:00–4:00 — One recommendation.** Start at the top of the ranked queue with `refresh` and `fix_ctr`; a human reviews every pick before any edit; `defer` the pages with nothing left to recover.

**4:00–5:00 — Why it matters.** It is decision-support, not automation — every claim is observed / measured / directional, and the whole thing reruns from `work/notebooks/capstone.ipynb`.

## 9. Two shareable cuts

**Social post (methodology-first):**

> Out of thousands of pages, which one should a content editor fix first? I ranked 30,000 pseudonymized pages with a transparent rule, then trained a logistic regression that had to beat it on clients it never saw. Result: the rule ranked below chance (Precision@50 0.26 vs a 0.39 base rate); the model reached 0.72 — and a human still makes every call. Built on the FlyRank ML Internship dataset.

**Employer 3-sentence summary:**

> I built a content-refresh ranking system on the FlyRank ML Internship dataset — 30,000 pseudonymized pages across 32 clients with 90-day search analytics. I encoded a transparent hand-written baseline, then trained and validated a logistic-regression model on a client-holdout split, lifting Precision@50 from 0.26 (below the base rate) to 0.72 and shipping a ranked, human-reviewed action playbook with reason codes. The work is deployed as a public research page with full reproducibility links.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed via nbconvert)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — and the paper is deployed with its URL in `submission/paper_url.txt`. Done.
